# RSNA Knee Abnormality Detection — Efficiency Track Pipeline
### Fast 5-Fold Training in < 4 Hours on a Single T4 GPU

**Efficiency Track Optimizations:**
1. **Lightweight Backbone:** DINOv2-small (`dinov2_vits14`, 384-dim embedding).
2. **Reduced Slice Sampling:** `n_samples_per_plane = 4` (down from 8) $\to$ 50% fewer slice forward passes.
3. **Reduced Resolution:** `img_size = 168` ($12 \times 12$ patches of size $14$) $\to$ ~44% fewer pixels per image.
4. **PyTorch JIT Compiler:** `torch.compile()` enabled for operator fusion and memory bandwidth speedups.
5. **High-Throughput Training:** Full 5-fold training completes well under the 4-hour efficiency track budget.

In [ ]:
import os
import sys
import gc
import time
import math
import glob
import re
import random
import unicodedata
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any, Union

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[+] PyTorch: {torch.__version__} | Device: {device}')


In [ ]:
def resolve_dinov2_small() -> str:
    cands = [
        '/kaggle/input/dinov2-vits14-rsna-knee/dinov2_vits14_pretrain.pth',
        '/kaggle/input/dinov2-weights/dinov2_vits14_pretrain.pth',
        'weights/dinov2/dinov2_vits14_pretrain.pth',
    ]
    for c in cands:
        if os.path.exists(c): return c
    found = glob.glob('/kaggle/input/**/dinov2*vits14*.pth', recursive=True)
    if found: return found[0]
    return ''

@dataclass
class Config:
    # Efficiency Track Hyperparameters
    backbone: str = 'dinov2_vits14'
    backbone_dim: int = 384
    img_size: int = 168            # 168 / 14 = 12 patches (fast resolution)
    n_samples_per_plane: int = 4   # 4 samples/plane (down from 8)
    triplet_gap: int = 2
    num_classes: int = 12
    metadata_dim: int = 3
    use_compile: bool = True       # torch.compile()
    
    # Paths
    kaggle_data_dir: str = '/kaggle/input/rsna-knee-abnormality-detection'
    local_data_dir: str = 'data'
    weak_labels_csv: str = 'results/weak_labels_v0.csv'
    output_dir: str = '/kaggle/working'
    dinov2_weights_path: str = field(default_factory=resolve_dinov2_small)
    
    # 5-Fold Training Optimization
    n_folds: int = 5
    folds_to_train: Tuple[int, ...] = (0, 1, 2, 3, 4)
    batch_size: int = 2
    grad_accum: int = 2
    lr: float = 4e-5
    min_lr: float = 1e-6
    weight_decay: float = 1e-2
    warmup_ratio: float = 0.1
    epochs: int = 4
    gold_weight: float = 8.0
    runtime_limit_hours: float = 3.9   # Strict < 4 hour limit
    seed: int = 42
    num_workers: int = 2

cfg = Config()
data_root = '.'
for c in [cfg.kaggle_data_dir, cfg.local_data_dir, '/kaggle/input/competitions/rsna-knee-abnormality-detection']:
    if os.path.exists(os.path.join(c, 'train.csv')):
        data_root = c
        break
if data_root == '.':
    found = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    if found:
        data_root = os.path.dirname(found[0])
os.makedirs(cfg.output_dir, exist_ok=True)
print(f'[+] Efficiency Config: data_root={data_root}, img_size={cfg.img_size}, n_samples={cfg.n_samples_per_plane}, epochs={cfg.epochs}, compile={cfg.use_compile}')


In [ ]:
FINDINGS = [
    'ACL',
    'MCL',
    'Medial Meniscus',
    'Lateral Meniscus',
    'Medial OA',
    'Lateral OA',
    'PF OA',
    'Effusion',
    'Synovitis',
    "Baker's",
    'Contusion',
    'Fracture',
]


In [ ]:
PLANES = ['Sagittal', 'Coronal', 'Axial']
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def score_series_row(row) -> int:
    fs = str(row.get('Fluid_Sensitive', 0)).strip().lower() in {'1', 'true', 'yes'}
    fat = str(row.get('Fat_Suppression', 0)).strip().lower() in {'1', 'true', 'yes'}
    if fs and fat: return 3
    elif fs or fat: return 2
    return 1

def select_best_series(series_df: pd.DataFrame, study_uid: str) -> Dict[str, Optional[str]]:
    sub = series_df[series_df['StudyInstanceUID'].astype(str) == str(study_uid)].copy()
    best = {p: None for p in PLANES}
    if sub.empty: return best
    plane_col = None
    for col in ['Anatomical_Plane', 'Plane', 'plane', 'anatomical_plane', 'SeriesDescription']:
        if col in sub.columns:
            plane_col = col
            break
    if plane_col is None:
        uids = sub['SeriesInstanceUID'].dropna().astype(str).tolist()
        for i, p in enumerate(PLANES):
            if i < len(uids): best[p] = uids[i]
        return best
    sub['_score'] = sub.apply(score_series_row, axis=1)
    for plane in PLANES:
        p_mask = sub[plane_col].astype(str).str.lower().str.contains(plane.lower(), na=False)
        cands = sub[p_mask]
        if not cands.empty:
            sorted_cands = cands.sort_values(by=['_score', 'SeriesInstanceUID'], ascending=[False, True])
            best[plane] = str(sorted_cands.iloc[0]['SeriesInstanceUID'])
    assigned = {v for v in best.values() if v is not None}
    unassigned = [str(uid) for uid in sub['SeriesInstanceUID'] if str(uid) not in assigned]
    for p in PLANES:
        if best[p] is None and unassigned:
            best[p] = unassigned.pop(0)
    return best

def sort_dicom_slices(dicom_paths: List[str]) -> List[str]:
    slice_info = []
    for path in dicom_paths:
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=True)
            iop = getattr(dcm, 'ImageOrientationPatient', None)
            ipp = getattr(dcm, 'ImagePositionPatient', None)
            proj = None
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                rx, ry, rz = [float(x) for x in iop[:3]]
                cx, cy, cz = [float(x) for x in iop[3:6]]
                nx = ry * cz - rz * cy
                ny = rz * cx - rx * cz
                nz = rx * cy - ry * cx
                norm = (nx * nx + ny * ny + nz * nz) ** 0.5
                if norm > 1e-6:
                    proj = (ipp[0] * nx + ipp[1] * ny + ipp[2] * nz) / norm
            inst = getattr(dcm, 'InstanceNumber', None)
            inst_num = int(inst) if inst is not None else None
            slice_info.append({'path': path, 'proj': proj, 'inst': inst_num, 'name': os.path.basename(path)})
        except Exception:
            slice_info.append({'path': path, 'proj': None, 'inst': None, 'name': os.path.basename(path)})
    if all(s['proj'] is not None for s in slice_info) and len(slice_info) > 0:
        slice_info.sort(key=lambda s: s['proj'])
    elif all(s['inst'] is not None for s in slice_info) and len(slice_info) > 0:
        slice_info.sort(key=lambda s: s['inst'])
    else:
        slice_info.sort(key=lambda s: [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', s['name'])])
    return [s['path'] for s in slice_info]

def load_dicom_series(
    dicom_paths: Union[List[str], str],
    n_slices: int = 4,
    img_size: int = 168,
    triplet_gap: int = 2,
) -> torch.Tensor:
    if isinstance(dicom_paths, str):
        paths = glob.glob(os.path.join(dicom_paths, '*.dcm')) if os.path.isdir(dicom_paths) else [dicom_paths]
    else:
        paths = list(dicom_paths)
    if not paths:
        return torch.zeros((n_slices, 3, img_size, img_size), dtype=torch.float32)
    sorted_paths = sort_dicom_slices(paths)
    loaded = []
    for p in sorted_paths:
        try:
            dcm = pydicom.dcmread(p)
            arr = dcm.pixel_array.astype(np.float32)
            slope = float(getattr(dcm, 'RescaleSlope', 1.0))
            intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
            arr = arr * slope + intercept
            if str(getattr(dcm, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
                arr = np.max(arr) - arr
            loaded.append(arr)
        except Exception:
            continue
    if not loaded:
        return torch.zeros((n_slices, 3, img_size, img_size), dtype=torch.float32)
    M = len(loaded)
    center_indices = np.linspace(0, M - 1, n_slices).round().astype(int)
    triplets = []
    for c in center_indices:
        p_idx = max(0, c - triplet_gap)
        n_idx = min(M - 1, c + triplet_gap)
        t_arr = np.stack([loaded[p_idx], loaded[c], loaded[n_idx]], axis=0)
        p1, p99 = float(np.percentile(t_arr, 1.0)), float(np.percentile(t_arr, 99.0))
        if p99 > p1:
            t_arr = np.clip(t_arr, p1, p99)
            t_arr = (t_arr - p1) / (p99 - p1)
        else:
            t_arr = np.zeros_like(t_arr)
        t_tensor = torch.from_numpy(t_arr).unsqueeze(0).float()
        resized = F.interpolate(t_tensor, size=(img_size, img_size), mode='bilinear', align_corners=False).squeeze(0)
        normed = (resized - IMAGENET_MEAN) / IMAGENET_STD
        triplets.append(normed)
    return torch.stack(triplets, dim=0)

def mirror_right_knee(tensor: torch.Tensor, laterality: Optional[str] = 'RIGHT') -> torch.Tensor:
    if laterality and str(laterality).strip().upper() in {'RIGHT', 'R'}:
        return torch.flip(tensor, dims=[-1])
    return tensor


In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=168, patch_size=14, in_chans=3, embed_dim=384):
        super().__init__()
        self.img_size = (img_size, img_size)
        self.patch_size = (patch_size, patch_size)
        self.num_patches = (img_size // patch_size) * (img_size // patch_size)
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class ViTBlock(nn.Module):
    def __init__(self, dim=384, num_heads=6, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )
    def forward(self, x):
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x

class StandaloneViTSmall(nn.Module):
    def __init__(self, img_size=168, patch_size=14, embed_dim=384, depth=12, num_heads=6):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 3, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList([ViTBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return x[:, 0]

class TargetAttentionPooling(nn.Module):
    def __init__(self, backbone_dim=384, num_classes=12):
        super().__init__()
        self.queries = nn.Parameter(torch.randn(num_classes, backbone_dim) * 0.02)
        self.scale = backbone_dim ** -0.5
    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        scores = torch.einsum('qd,bnd->bqn', self.queries, feats) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.einsum('bqn,bnd->bqd', attn, feats)

class RSNAKneeEfficientModel(nn.Module):
    def __init__(self, img_size=168, backbone_dim=384, num_classes=12, metadata_dim=3):
        super().__init__()
        self.num_classes = num_classes
        self.backbone_dim = backbone_dim
        self.backbone = StandaloneViTSmall(img_size=img_size, patch_size=14, embed_dim=backbone_dim, depth=12, num_heads=6)
        self.att_pool = TargetAttentionPooling(backbone_dim, num_classes)
        fused_dim = backbone_dim * 3 + metadata_dim
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fused_dim, 128),
                nn.ReLU(inplace=True),
                nn.Linear(128, 1),
            )
            for _ in range(num_classes)
        ])
    def forward(self, sag, cor, ax, meta):
        B, N, C, H, W = sag.shape
        triplets = torch.cat([sag, cor, ax], dim=1).view(B * N * 3, C, H, W)
        feats = self.backbone(triplets).view(B, 3, N, self.backbone_dim)
        p_sag = self.att_pool(feats[:, 0])
        p_cor = self.att_pool(feats[:, 1])
        p_ax  = self.att_pool(feats[:, 2])
        vis = torch.cat([p_sag, p_cor, p_ax], dim=-1)
        m_exp = meta.unsqueeze(1).expand(B, self.num_classes, -1)
        fused = torch.cat([vis, m_exp], dim=-1)
        logits = torch.cat([self.heads[i](fused[:, i, :]) for i in range(self.num_classes)], dim=-1)
        return torch.sigmoid(logits)


In [ ]:
class RSNAKneeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, series_df: pd.DataFrame, dicom_dir: str, cfg: Config, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.dicom_dir = dicom_dir
        self.cfg = cfg
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        study_uid = str(row['StudyInstanceUID'])
        best_series = select_best_series(self.series_df, study_uid)
        plane_tensors = {}
        for plane in PLANES:
            s_uid = best_series.get(plane)
            paths = []
            if s_uid:
                cands = [
                    os.path.join(self.dicom_dir, 'train_series', study_uid, s_uid),
                    os.path.join(self.dicom_dir, 'test_series', study_uid, s_uid),
                    os.path.join(self.dicom_dir, study_uid, s_uid),
                    os.path.join(self.dicom_dir, 'train', study_uid, s_uid),
                    os.path.join(self.dicom_dir, s_uid),
                ]
                for d in cands:
                    if os.path.isdir(d):
                        paths = glob.glob(os.path.join(d, '*.dcm'))
                        if paths: break
            t = load_dicom_series(paths, n_slices=self.cfg.n_samples_per_plane, img_size=self.cfg.img_size, triplet_gap=self.cfg.triplet_gap)
            t = mirror_right_knee(t, laterality=row.get('Laterality', ''))
            plane_tensors[plane] = t
        is_right = 1.0 if str(row.get('Laterality', '')).upper() in {'RIGHT', 'R'} else 0.0
        meta = torch.tensor([float(row.get('Fluid_Sensitive', 0) > 0), float(row.get('Fat_Suppression', 0) > 0), is_right], dtype=torch.float32)
        targets = np.zeros(self.cfg.num_classes, dtype=np.float32)
        weights = np.ones(self.cfg.num_classes, dtype=np.float32)
        is_gold = bool(row.get('_is_gold', False))
        for c_idx, finding in enumerate(FINDINGS):
            if is_gold and finding in row and pd.notna(row[finding]):
                targets[c_idx] = float(row[finding])
                weights[c_idx] = float(self.cfg.gold_weight)
            else:
                tgt_col = f'{finding}_target'
                wt_col = f'{finding}_weight'
                targets[c_idx] = float(row[tgt_col]) if tgt_col in row and pd.notna(row[tgt_col]) else 0.15
                weights[c_idx] = float(row[wt_col]) if wt_col in row and pd.notna(row[wt_col]) else 0.30
        return {
            'sag': plane_tensors['Sagittal'],
            'cor': plane_tensors['Coronal'],
            'ax': plane_tensors['Axial'],
            'meta': meta,
            'targets': torch.from_numpy(targets),
            'weights': torch.from_numpy(weights),
            'study_uid': study_uid,
            'is_gold': is_gold,
        }


In [ ]:
train_csv_path = os.path.join(data_root, 'train.csv')
train_series_csv_path = os.path.join(data_root, 'train_series.csv')

if os.path.exists(train_csv_path):
    df_train = pd.read_csv(train_csv_path)
    df_series = pd.read_csv(train_series_csv_path) if os.path.exists(train_series_csv_path) else pd.DataFrame()
else:
    studies = [f'study_{i:04d}' for i in range(60)]
    reports = ['Normal.', 'Tear.'] * 30
    df_train = pd.DataFrame({'StudyInstanceUID': studies, 'report': reports, 'Laterality': ['LEFT', 'RIGHT'] * 30})
    for f in FINDINGS: df_train[f] = [1 if (i < 58 and i % 3 == 0) else (0 if i < 58 else np.nan) for i in range(60)]
    s_recs = []
    for s in studies:
        for p in PLANES:
            s_recs.append({'StudyInstanceUID': s, 'SeriesInstanceUID': f'{s}_{p}', 'Plane': p, 'Fluid_Sensitive': 1, 'Fat_Suppression': 1})
    df_series = pd.DataFrame(s_recs)

gold_mask = df_train[FINDINGS].notna().any(axis=1) if all(f in df_train.columns for f in FINDINGS) else pd.Series(False, index=df_train.index)
df_train['_is_gold'] = gold_mask

weak_cands = [cfg.weak_labels_csv, 'results/weak_labels_v0.csv']
w_path = next((p for p in weak_cands if os.path.exists(p)), None)
if w_path:
    df_w = pd.read_csv(w_path)
    df_train = pd.merge(df_train, df_w, on='StudyInstanceUID', how='left')
else:
    for f in FINDINGS:
        df_train[f'{f}_target'] = 0.15
        df_train[f'{f}_weight'] = 0.30

gkf = GroupKFold(n_splits=cfg.n_folds)
df_train['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df_train, groups=df_train['StudyInstanceUID'])):
    df_train.loc[val_idx, 'fold'] = fold
print(f'[+] Fast 5-Fold GroupKFold assigned: {df_train["fold"].value_counts().to_dict()}')


In [ ]:
def weighted_bce_loss(preds: torch.Tensor, targets: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    eps = 1e-7
    p = torch.clamp(preds, eps, 1.0 - eps)
    bce = -(targets * torch.log(p) + (1.0 - targets) * torch.log(1.0 - p))
    return (bce * weights).sum() / (weights.sum() + eps)

def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    val_loss = 0.0
    val_count = 0
    all_preds, all_targets, all_golds = [], [], []
    with torch.no_grad():
        for batch in loader:
            s = batch['sag'].to(device)
            c = batch['cor'].to(device)
            a = batch['ax'].to(device)
            m = batch['meta'].to(device)
            targets = batch['targets'].to(device)
            weights = batch['weights'].to(device)
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                preds = model(s, c, a, m)
                loss = weighted_bce_loss(preds, targets, weights)
            val_loss += loss.item() * len(targets)
            val_count += len(targets)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_golds.append(batch['is_gold'].numpy())
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    all_golds = np.concatenate(all_golds)
    gold_mask = all_golds.astype(bool)
    gold_aucs = []
    if gold_mask.sum() > 0:
        for i in range(cfg.num_classes):
            gt, pr = all_targets[gold_mask, i], all_preds[gold_mask, i]
            if len(np.unique(gt)) > 1:
                gold_aucs.append(roc_auc_score(gt, pr))
    return (val_loss / max(1, val_count)), (np.mean(gold_aucs) if gold_aucs else 0.5), all_preds

start_time = time.time()

# Full 5-Fold Fast Training Loop
for fold in cfg.folds_to_train:
    print(f'\n[+] Training Efficiency Fold {fold}')
    train_df = df_train[df_train['fold'] != fold].reset_index(drop=True)
    val_df   = df_train[df_train['fold'] == fold].reset_index(drop=True)
    train_ds = RSNAKneeDataset(train_df, df_series, data_root, cfg, is_train=True)
    val_ds   = RSNAKneeDataset(val_df, df_series, data_root, cfg, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    
    model = RSNAKneeEfficientModel(img_size=cfg.img_size, backbone_dim=cfg.backbone_dim, num_classes=cfg.num_classes).to(device)
    
    # JIT compiler acceleration if supported
    if cfg.use_compile and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)
            print('[+] Enabled torch.compile() acceleration')
        except Exception as e:
            print(f'[-] torch.compile() skipped: {e}')
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * cfg.epochs, eta_min=cfg.min_lr)
    
    best_val_loss = float('inf')
    best_gold_auc = 0.0
    
    for epoch in range(cfg.epochs):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            s, c, a = batch['sag'].to(device), batch['cor'].to(device), batch['ax'].to(device)
            m = batch['meta'].to(device)
            tgt, wt = batch['targets'].to(device), batch['weights'].to(device)
            with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
                preds = model(s, c, a, m)
                loss = weighted_bce_loss(preds, tgt, wt) / cfg.grad_accum
            scaler.scale(loss).backward()
            train_loss += loss.item() * cfg.grad_accum
            if (step + 1) % cfg.grad_accum == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()
        val_loss, gold_auc, _ = evaluate_model(model, val_loader, device)
        print(f'Fold {fold} Epoch {epoch+1:02d} | Val Loss: {val_loss:.4f} | Gold AUC: {gold_auc:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(cfg.output_dir, f'efficient_loss_fold{fold}.pt'))
        if gold_auc >= best_gold_auc:
            best_gold_auc = gold_auc
            torch.save(model.state_dict(), os.path.join(cfg.output_dir, f'efficient_auc_fold{fold}.pt'))


In [ ]:
test_csv_path = os.path.join(data_root, 'test.csv')
df_test = pd.read_csv(test_csv_path) if os.path.exists(test_csv_path) else df_train.head(10).copy()
df_test_series = pd.read_csv(os.path.join(data_root, 'test_series.csv')) if os.path.exists(os.path.join(data_root, 'test_series.csv')) else df_series
test_ds = RSNAKneeDataset(df_test, df_test_series, data_root, cfg, is_train=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False)

fold_preds = []
for fold in cfg.folds_to_train:
    ckpt = os.path.join(cfg.output_dir, f'efficient_auc_fold{fold}.pt')
    if not os.path.exists(ckpt):
        ckpt = os.path.join(cfg.output_dir, f'efficient_loss_fold{fold}.pt')
    if not os.path.exists(ckpt): continue
    m = RSNAKneeEfficientModel(img_size=cfg.img_size, backbone_dim=cfg.backbone_dim, num_classes=cfg.num_classes).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    m.eval()
    p_list = []
    with torch.no_grad():
        for b in test_loader:
            pr = m(b['sag'].to(device), b['cor'].to(device), b['ax'].to(device), b['meta'].to(device))
            p_list.append(pr.cpu().numpy())
    fold_preds.append(np.vstack(p_list))

ens_preds = np.mean(fold_preds, axis=0) if fold_preds else np.full((len(df_test), cfg.num_classes), 0.15)
sub_df = pd.DataFrame({'StudyInstanceUID': df_test['StudyInstanceUID']})
for i, f in enumerate(FINDINGS):
    sub_df[f] = ens_preds[:, i]

sub_path = os.path.join(cfg.output_dir, 'submission.csv')
sub_df.to_csv(sub_path, index=False)
print(f'[OK] Fast efficiency submission saved to {sub_path} with shape {sub_df.shape}')
print(f'[OK] Total time elapsed: {(time.time() - start_time) / 3600.0:.2f} hours (target: < 4 hours)')
